# Bengaluru House Price Prediction
### EDA · Data Cleaning · Model Training

> Run all cells top-to-bottom. Four files will be saved at the end:
> `RF_model.pkl`, `encoder.pkl`, `feature_columns.pkl`, `Cleaned_df.csv`

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import joblib

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

## 2. Load Data

In [ ]:
df = pd.read_csv("Bengaluru_House_Data.csv")
print("Shape:", df.shape)
df.head()

In [ ]:
df.info()

## 3. Drop Unwanted Columns

In [ ]:
df = df.drop(["area_type", "availability", "society", "balcony"], axis=1)
df.head()

## 4. Handle Missing Values

In [ ]:
df.isnull().sum()

In [ ]:
# Location → fill with most frequent value
df["location"] = df["location"].fillna("Sarjapur  Road")

# Size → fill with most common
df["size"] = df["size"].fillna("2 BHK")

# Bath → fill with median, cast to int
med = df["bath"].median()
df["bath"] = df["bath"].fillna(med).astype(int)

print("Null values after fix:")
df.isnull().sum()

## 5. Feature Engineering

In [ ]:
# Extract BHK number from size column  e.g. '2 BHK' → 2
def clean_size(size):
    return int(size.split()[0])

df["bhk"] = df["size"].apply(clean_size)
print(df["bhk"].unique())

In [ ]:
# Rare locations (≤10 listings) → grouped as 'Others'
loc_counts       = df["location"].value_counts()
loc_less_than_10 = loc_counts[loc_counts <= 10]

def clean_location(location):
    return "Others" if location in loc_less_than_10 else location

df["location"] = df["location"].apply(clean_location)
df["location"].value_counts().head(10)

In [ ]:
# Fix total_sqft — handle ranges like '1133-1384' by taking the average
def clean_sqft(sqft):
    parts = str(sqft).split("-")
    if len(parts) == 2:
        try:
            return (float(parts[0]) + float(parts[1])) / 2   # ✅ correct average
        except:
            return None
    try:
        return float(parts[0])
    except:
        return None

df["total_sqft"] = df["total_sqft"].apply(clean_sqft)
df["total_sqft"] = df["total_sqft"].fillna(round(df["total_sqft"].mean()))
df["total_sqft"].head()

## 6. Outlier Removal

In [ ]:
# Price per sqft — used only for outlier detection
df["price_per_sqft"] = df["price"] * 100000 / df["total_sqft"]

# Remove rows where sqft per BHK < 300 (unrealistic)
df = df[df["total_sqft"] / df["bhk"] >= 300]

df.describe()

In [ ]:
# Cap BHK at 6
df = df[df["bhk"] <= 6]

sns.boxplot(x="bhk", data=df)
plt.title("BHK Distribution")
plt.show()

In [ ]:
# Bath must be < bhk + 2  (realistic plumbing rule)
df = df[df["bath"] < df["bhk"] + 2]
print("Shape after bath filter:", df.shape)

In [ ]:
# IQR filter on price_per_sqft  (threshold = 0.5 × IQR)
q1    = df["price_per_sqft"].quantile(0.25)
q3    = df["price_per_sqft"].quantile(0.75)
IQR   = q3 - q1
lower = q1 - 0.5 * IQR
upper = q3 + 0.5 * IQR

df = df[(df["price_per_sqft"] >= lower) & (df["price_per_sqft"] <= upper)]

sns.boxplot(x="price_per_sqft", data=df)
plt.title("Price Per Sqft — after outlier removal")
plt.show()

## 7. Final Cleanup

In [ ]:
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)   # drop=True avoids extra index column

# Save full cleaned data BEFORE dropping helper columns
df.to_csv("Cleaned_df.csv", index=False)
print("✅ Cleaned_df.csv saved — shape:", df.shape)

# Drop columns not needed for training
df = df.drop(["price_per_sqft", "size"], axis=1)
df.head()

## 8. Encode & Split

In [ ]:
encoder = LabelEncoder()
df["encoded_loc"] = encoder.fit_transform(df["location"])

X = df.drop(["location", "price"], axis=1)
y = df["price"]

print("Feature columns:", list(X.columns))
X.head()

In [ ]:
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.3, random_state=42)
print("Train:", Xtrain.shape, " | Test:", Xtest.shape)

## 9. Baseline Models

In [ ]:
LR = LinearRegression()
RF = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)

LR.fit(Xtrain, ytrain)
RF.fit(Xtrain, ytrain)

print("Linear Regression:")
print(f"  Train R²: {LR.score(Xtrain, ytrain):.4f}  |  Test R²: {LR.score(Xtest, ytest):.4f}")

print("\nRandom Forest (baseline):")
print(f"  Train R²: {RF.score(Xtrain, ytrain):.4f}  |  Test R²: {RF.score(Xtest, ytest):.4f}")

**RF outperforms LR → tune it with GridSearchCV**

## 10. Hyperparameter Tuning

In [ ]:
model  = RandomForestRegressor(random_state=42)
params = {
    "n_estimators": [100, 150, 200, 250, 300],
    "max_depth":    [5, 6, 7]
}
grid = GridSearchCV(estimator=model, param_grid=params, cv=5, n_jobs=-1, verbose=1)
grid.fit(Xtrain, ytrain)

print("Best params :", grid.best_params_)
print("Best CV R²  :", round(grid.best_score_, 4))

In [ ]:
ypred = grid.predict(Xtest)

print("Tuned Random Forest:")
print(f"  Train R²: {grid.score(Xtrain, ytrain):.4f}  |  Test R²: {grid.score(Xtest, ytest):.4f}")
print(f"  R2 Score: {r2_score(ytest, ypred):.4f}")
print(f"  MAE     : {mean_absolute_error(ytest, ypred):.4f}")

## 11. Save Artifacts

In [ ]:
# ✅ Save best tuned model  (NOT the base untuned 'model' variable)
joblib.dump(grid.best_estimator_, "RF_model.pkl")

# ✅ Save encoder  (needed by app.py to encode user-selected location)
joblib.dump(encoder, "encoder.pkl")

# ✅ Save feature column order  (prevents column-mismatch bugs in app.py)
joblib.dump(list(X.columns), "feature_columns.pkl")

print("✅ RF_model.pkl        — tuned RandomForest")
print("✅ encoder.pkl         — LabelEncoder for location")
print("✅ feature_columns.pkl — ['total_sqft', 'bath', 'bhk', 'encoded_loc']")
print("✅ Cleaned_df.csv      — cleaned dataset for location dropdown")